In [29]:
import json
from pathlib import Path
from typing import List
from tqdm import tqdm

import requests

In [30]:
SA_QUESTION_FILES = {
    "DoS":   Path("DoS_sa_qa/questions/dos_sa_questions.json"),
    "Fuzzy": Path("Fuzzy_sa_qa/questions/fuzzy_sa_questions.json"),
    "Gear":  Path("Gear_sa_qa/questions/gear_sa_questions.json"),
    "RPM":   Path("RPM_sa_qa/questions/rpm_sa_questions.json"),
}

SELECTED_DATASETS = ["DoS", "Fuzzy", "Gear", "RPM"]

In [31]:
OLLAMA_BASE_URL = "http://127.0.0.1:11434"

BASE_MODEL_ID = "glm4:latest"
USE_PRETEST_ADAPTATION = True
ADAPTED_MODEL_ID = "glm4_can_sa_adapted:latest"

# Adaptation settings (fraction is primary; other caps are safety guards)
# Set to 0.25 for 25% or 0.50 for 50% from each dataset (attack type).
ADAPT_FRACTION_PER_DATASET = 0.50
ADAPT_MAX_EXAMPLES = 999999
ADAPT_MAX_PER_DATASET = 999999
ADAPT_MAX_CONTEXT_CHARS = 220
ADAPT_MAX_SYSTEM_CHARS = 42000
ADAPT_SEED = 42

MODEL_ID = ADAPTED_MODEL_ID if USE_PRETEST_ADAPTATION else BASE_MODEL_ID
MODEL_TAG = MODEL_ID.replace(":", "_").replace("-", "_")

print(f"Ollama URL: {OLLAMA_BASE_URL}")
print(f"Base model: {BASE_MODEL_ID}")
print(f"Pre-test adaptation enabled: {USE_PRETEST_ADAPTATION}")
print(f"Adaptation fraction per dataset: {ADAPT_FRACTION_PER_DATASET:.0%}")
print(f"Adaptation max examples cap: {ADAPT_MAX_EXAMPLES}")
print(f"Active model: {MODEL_ID}")




Ollama URL: http://127.0.0.1:11434
Base model: glm4:latest
Pre-test adaptation enabled: True
Adaptation fraction per dataset: 50%
Adaptation max examples cap: 999999
Active model: glm4_can_sa_adapted:latest


In [32]:
import re

SA_OUTPUT_TYPE = {
    "attack_ratio": "percentage",
    "attack_pattern": "short_text",
    "dominant_id": "can_id",
    "id_concentration": "short_text",
    "fps": "number",
    "traffic_shift": "short_text",
    "max_gap": "number",
    "timing_cause": "short_text",
    "dominant_var": "number",
    "payload_behavior": "short_text",
    "dlc_mode": "integer",
    "dlc_behavior": "short_text",
    "missing_count": "integer",
    "critical_behavior": "short_text",
    "out_of_range_id_count": "integer",
    "plausibility_relation": "short_text",
    "duplicate_count": "integer",
    "timestamp_cause": "short_text",
    "attack_type": "short_text",
    "evidence_pattern": "short_text",
}

SA_TYPE_HINT = {
    "percentage": "Answer with a percentage only, e.g., 12.5%.",
    "can_id": "Answer with one CAN ID only, uppercase hex format, e.g., 0x1AF.",
    "integer": "Answer with one non-negative integer only.",
    "number": "Answer with one numeric value only.",
    "short_text": "Answer with a short phrase only (1-4 words), not a full sentence.",
}


def build_sa_prompt_text(context: str, question: str, output_type: str = "short_text", strict_mode: bool = False) -> str:
    """Short-answer prompt that returns answer + explanation in two lines."""
    system_prompt = (
        "You are a CAN-bus anomaly analysis assistant. "
        "Read a CAN time-window log and answer one short-answer question. "
        "Think using timestamp patterns, ID frequency, gaps, DLC, payload behavior, flags, and baseline cues internally. "
        "Provide output in exactly two lines only. "
        "Line 1 must be only the direct final answer. "
        "Line 2 must be a brief explanation with at most 10 words."
    )

    type_hint = SA_TYPE_HINT.get(output_type, SA_TYPE_HINT["short_text"])
    strict_tail = (
        "\nType enforcement: strictly follow the answer type hint only."
        if strict_mode else ""
    )

    return (
        f"{system_prompt}\n\n"
        "Analyze the following CAN bus time-window log and answer the question.\n\n"
        f"CAN log:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Answer type hint:\n{type_hint}{strict_tail}\n\n"
        "Output exactly two lines:\n"
        "Line 1: Answer only\n"
        "Line 2: Brief explanation (max 10 words)"
    )


def normalize_text(text: str) -> str:
    return " ".join(text.strip().split()).lower() if text else ""


def _sanitize_answer(text: str) -> tuple[str, str]:
    """Returns (answer, explanation) tuple."""
    lines = [line.strip() for line in (text or "").strip().splitlines() if line.strip()]
    if not lines:
        return "", ""

    answer = lines[0].strip("\"'`")
    explanation = lines[1].strip("\"'`") if len(lines) > 1 else ""
    return answer, explanation


def _is_valid_answer_format(ans: str, output_type: str) -> bool:
    if not ans:
        return False

    if output_type == "can_id":
        return re.fullmatch(r"0x[0-9A-F]+", ans) is not None

    if output_type == "integer":
        return re.fullmatch(r"\d+", ans) is not None

    if output_type == "percentage":
        if not ans.endswith("%"):
            return False
        try:
            float(ans[:-1])
            return True
        except Exception:
            return False

    if output_type == "number":
        try:
            float(ans)
            return True
        except Exception:
            return False

    words = ans.split()
    return 1 <= len(words) <= 4


def _extract_ollama_text(response_json: dict) -> str:
    if isinstance(response_json, dict):
        if isinstance(response_json.get("response"), str):
            return response_json["response"]
        if "message" in response_json and isinstance(response_json["message"], dict):
            content = response_json["message"].get("content")
            if isinstance(content, str):
                return content
    return ""


def query_sa_llm(context: str, question: str, sa_type: str = "", max_new_tokens: int = 48) -> tuple[str, str]:
    """Returns (answer, explanation) tuple."""
    output_type = SA_OUTPUT_TYPE.get(sa_type, "short_text")

    def _run_once(strict_mode: bool) -> tuple[str, str]:
        prompt_text = build_sa_prompt_text(
            context, question, output_type=output_type, strict_mode=strict_mode
        )
        payload = {
            "model": MODEL_ID,
            "prompt": prompt_text,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": max_new_tokens,
            },
        }
        resp = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json=payload,
            timeout=180,
        )
        resp.raise_for_status()
        completion = _extract_ollama_text(resp.json())
        return _sanitize_answer(completion)

    ans, expl = _run_once(strict_mode=False)
    if _is_valid_answer_format(ans, output_type):
        return ans, expl

    retry_ans, retry_expl = _run_once(strict_mode=True)
    if retry_ans:
        return retry_ans, retry_expl
    return ans, expl


def load_questions(path: Path) -> List[dict]:
    """Load questions from .json (list) or .jsonl."""
    if not path.exists():
        return []
    if path.suffix == ".json":
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records




In [33]:
from collections import defaultdict
import random
import math

# Lightweight pre-test adaptation: create an Ollama-derived model with task-specific system guidance.
# This is prompt-level adaptation, not gradient/weight fine-tuning.

def _truncate_text(text: str, max_chars: int = 220) -> str:
    text = " ".join((text or "").split())
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + " ..."


def _extract_valid_records(records: list[dict]) -> list[dict]:
    return [rec for rec in records if rec.get("ground_truth", rec.get("answer", ""))]


def build_dataset_splits(fraction_per_dataset: float = 0.50, seed: int = 42) -> dict:
    """Deterministic split by qa_id per dataset: adapt(train) vs eval(holdout)."""
    frac = min(max(fraction_per_dataset, 0.01), 0.99)
    split_map = {}

    for ds_name in SELECTED_DATASETS:
        q_path = SA_QUESTION_FILES.get(ds_name)
        if not q_path or not q_path.exists():
            split_map[ds_name] = {
                "total_valid": 0,
                "adapt_records": [],
                "eval_records": [],
            }
            continue

        records = load_questions(q_path)
        valid_records = _extract_valid_records(records)

        # Stable ordering by qa_id before shuffle for deterministic partitioning.
        valid_records = sorted(valid_records, key=lambda r: str(r.get("qa_id", "")))

        ds_seed = seed + sum(ord(ch) for ch in ds_name)
        rng = random.Random(ds_seed)
        rng.shuffle(valid_records)

        total_valid = len(valid_records)
        if total_valid <= 1:
            adapt_n = total_valid
        else:
            # Ensure we keep a non-empty holdout for unseen evaluation.
            adapt_n = math.ceil(total_valid * frac)
            adapt_n = min(max(1, adapt_n), total_valid - 1)

        adapt_records = valid_records[:adapt_n]
        eval_records = valid_records[adapt_n:]

        split_map[ds_name] = {
            "total_valid": total_valid,
            "adapt_records": adapt_records,
            "eval_records": eval_records,
        }

    return split_map


def _build_adaptation_examples_from_splits(
    split_map: dict,
    max_per_dataset: int = 300,
    max_total: int = 120,
    seed: int = 42,
):
    rng = random.Random(seed)
    all_examples = []
    dataset_stats = {}

    for ds_name in SELECTED_DATASETS:
        info = split_map.get(ds_name, {})
        adapt_records = list(info.get("adapt_records", []))

        if not adapt_records:
            dataset_stats[ds_name] = {
                "adapt_pool": 0,
                "adapt_target": 0,
                "adapt_used": 0,
            }
            continue

        target_n = min(max_per_dataset, len(adapt_records)) if max_per_dataset > 0 else len(adapt_records)

        by_type = defaultdict(list)
        for rec in adapt_records:
            by_type[rec.get("sa_type", "short_text")].append(rec)

        for recs in by_type.values():
            rng.shuffle(recs)

        # Round-robin by SA type for better coverage.
        picked = []
        type_keys = list(by_type.keys())
        idx = {k: 0 for k in type_keys}
        while len(picked) < target_n and type_keys:
            progressed = False
            for k in list(type_keys):
                recs = by_type[k]
                if idx[k] < len(recs):
                    picked.append(recs[idx[k]])
                    idx[k] += 1
                    progressed = True
                    if len(picked) >= target_n:
                        break
                else:
                    type_keys.remove(k)
            if not progressed:
                break

        dataset_stats[ds_name] = {
            "adapt_pool": len(adapt_records),
            "adapt_target": target_n,
            "adapt_used": len(picked),
        }

        for rec in picked:
            all_examples.append(
                {
                    "dataset": ds_name,
                    "sa_type": rec.get("sa_type", "short_text"),
                    "question": rec.get("question", ""),
                    "answer": rec.get("ground_truth", rec.get("answer", "")),
                    "context": _truncate_text(rec.get("context", ""), ADAPT_MAX_CONTEXT_CHARS),
                }
            )

    rng.shuffle(all_examples)
    return all_examples[:max_total], dataset_stats


def _build_adaptation_system_prompt(examples, max_chars: int = 42000) -> str:
    rules = (
        "You are a CAN-bus anomaly short-answer model. "
        "Return only the final short answer with no explanation. "
        "Use CAN-log evidence such as timestamps, ID frequency, DLC, payload behavior, and flags. "
        "For numeric types, output only the numeric value. "
        "For CAN IDs, output uppercase hex like 0x1AF."
    )

    lines = [rules, "", "Calibration examples:"]
    for i, ex in enumerate(examples, start=1):
        block = [
            f"E{i} | ds={ex['dataset']} | type={ex['sa_type']}",
            f"Q: {_truncate_text(ex['question'], 220)}",
            f"A: {_truncate_text(str(ex['answer']), 80)}",
            f"C: {ex['context']}",
            "",
        ]

        candidate = "\n".join(lines + block)
        if len(candidate) > max_chars:
            break
        lines.extend(block)

    return "\n".join(lines)


def ensure_pretest_adapted_model(split_map: dict | None = None) -> None:
    global MODEL_ID, MODEL_TAG

    if not USE_PRETEST_ADAPTATION:
        print("[INFO] Pre-test adaptation disabled; using base model.")
        return

    if split_map is None:
        split_map = build_dataset_splits(
            fraction_per_dataset=ADAPT_FRACTION_PER_DATASET,
            seed=ADAPT_SEED,
        )

    examples, dataset_stats = _build_adaptation_examples_from_splits(
        split_map=split_map,
        max_per_dataset=ADAPT_MAX_PER_DATASET,
        max_total=ADAPT_MAX_EXAMPLES,
        seed=ADAPT_SEED,
    )

    for ds_name in SELECTED_DATASETS:
        info = split_map.get(ds_name, {})
        st = dataset_stats.get(ds_name, {})
        print(
            f"[INFO] {ds_name}: total_valid={info.get('total_valid', 0)} "
            f"adapt_pool={len(info.get('adapt_records', []))} "
            f"holdout_eval={len(info.get('eval_records', []))} "
            f"adapt_used={st.get('adapt_used', 0)}"
        )

    if not examples:
        MODEL_ID = BASE_MODEL_ID
        MODEL_TAG = MODEL_ID.replace(":", "_").replace("-", "_")
        print("[WARN] No adaptation examples found; using base model.")
        return

    current_examples = examples
    min_examples = min(12, len(current_examples))

    while current_examples:
        system_prompt = _build_adaptation_system_prompt(
            current_examples,
            max_chars=ADAPT_MAX_SYSTEM_CHARS,
        )

        payload = {
            "model": ADAPTED_MODEL_ID,
            "from": BASE_MODEL_ID,
            "system": system_prompt,
            "parameters": {
                "temperature": 0,
                "num_predict": 24,
            },
            "stream": False,
        }

        try:
            resp = requests.post(f"{OLLAMA_BASE_URL}/api/create", json=payload, timeout=600)
            resp.raise_for_status()
            MODEL_ID = ADAPTED_MODEL_ID
            MODEL_TAG = MODEL_ID.replace(":", "_").replace("-", "_")
            print(f"[INFO] Pre-test adapted model ready: {MODEL_ID}")
            print(f"[INFO] Adaptation examples requested after caps: {len(examples)}")
            print(f"[INFO] Adaptation examples used in final create call: {len(current_examples)}")
            return
        except Exception as e:
            if len(current_examples) <= min_examples:
                MODEL_ID = BASE_MODEL_ID
                MODEL_TAG = MODEL_ID.replace(":", "_").replace("-", "_")
                print(f"[WARN] Adaptation failed ({e}); falling back to base model: {MODEL_ID}")
                return

            new_n = max(min_examples, len(current_examples) // 2)
            print(f"[WARN] Adaptation create failed with {len(current_examples)} examples; retrying with {new_n}.")
            current_examples = current_examples[:new_n]



In [34]:
split_map = build_dataset_splits(
    fraction_per_dataset=ADAPT_FRACTION_PER_DATASET,
    seed=ADAPT_SEED,
)

ensure_pretest_adapted_model(split_map=split_map)
is_adapted_model = (MODEL_ID == ADAPTED_MODEL_ID)
print(f"[INFO] Using adapted model: {is_adapted_model}")
print(f"[INFO] Answer generation uses model: {MODEL_ID}")
print("[INFO] Evaluation runs on holdout questions only (unseen during adaptation).")

for ds_name in SELECTED_DATASETS:
    info = split_map.get(ds_name, {})
    questions = info.get("eval_records", [])

    if not questions:
        print(f"[WARN] {ds_name}: no holdout questions available, skip.")
        continue

    print(
        f"[INFO] {ds_name}: total_valid={info.get('total_valid', 0)} "
        f"adapt={len(info.get('adapt_records', []))} eval_holdout={len(questions)}"
    )

    q_path = SA_QUESTION_FILES[ds_name]
    out_dir = q_path.parent.parent / "llm_answers"
    out_dir.mkdir(parents=True, exist_ok=True)
    ans_path = out_dir / f"{ds_name.lower()}_sa_answers_{MODEL_TAG}.jsonl"

    ans_path.write_text("", encoding="utf-8")

    with ans_path.open("a", encoding="utf-8") as f:
        for rec in tqdm(questions, desc=f"{ds_name} SA holdout answering ({MODEL_ID})"):
            qa_id = rec.get("qa_id")
            metadata = rec.get("metadata", {})
            dataset = metadata.get("dataset", ds_name)
            context = rec["context"]
            question = rec["question"]
            gt = rec.get("ground_truth", rec.get("answer", ""))

            pred, pred_expl = query_sa_llm(context, question, sa_type=rec.get("sa_type", ""))

            answer_rec = {
                "qa_id": qa_id,
                "dataset": dataset,
                "sa_type": rec.get("sa_type"),
                "model": MODEL_ID,
                "llm_answer": pred,
                "llm_explanation": pred_expl,
                "ground_truth": gt,
                "is_exact_match": normalize_text(pred) == normalize_text(gt) if gt else False,
            }
            f.write(json.dumps(answer_rec, ensure_ascii=False) + "\n")

    print(f"[INFO] {ds_name}: holdout answers+explanations saved to {ans_path}")



[INFO] DoS: total_valid=916 adapt_pool=458 holdout_eval=458 adapt_used=458
[INFO] Fuzzy: total_valid=959 adapt_pool=480 holdout_eval=479 adapt_used=480
[INFO] Gear: total_valid=1110 adapt_pool=555 holdout_eval=555 adapt_used=555
[INFO] RPM: total_valid=1155 adapt_pool=578 holdout_eval=577 adapt_used=578
[INFO] Pre-test adapted model ready: glm4_can_sa_adapted:latest
[INFO] Adaptation examples requested after caps: 2071
[INFO] Adaptation examples used in final create call: 2071
[INFO] Using adapted model: True
[INFO] Answer generation uses model: glm4_can_sa_adapted:latest
[INFO] Evaluation runs on holdout questions only (unseen during adaptation).
[INFO] DoS: total_valid=916 adapt=458 eval_holdout=458


DoS SA holdout answering (glm4_can_sa_adapted:latest): 100%|██████████| 458/458 [1:09:41<00:00,  9.13s/it]


[INFO] DoS: holdout answers+explanations saved to DoS_sa_qa\llm_answers\dos_sa_answers_glm4_can_sa_adapted_latest.jsonl
[INFO] Fuzzy: total_valid=959 adapt=480 eval_holdout=479


Fuzzy SA holdout answering (glm4_can_sa_adapted:latest): 100%|██████████| 479/479 [1:11:28<00:00,  8.95s/it]


[INFO] Fuzzy: holdout answers+explanations saved to Fuzzy_sa_qa\llm_answers\fuzzy_sa_answers_glm4_can_sa_adapted_latest.jsonl
[INFO] Gear: total_valid=1110 adapt=555 eval_holdout=555


Gear SA holdout answering (glm4_can_sa_adapted:latest): 100%|██████████| 555/555 [1:06:28<00:00,  7.19s/it]


[INFO] Gear: holdout answers+explanations saved to Gear_sa_qa\llm_answers\gear_sa_answers_glm4_can_sa_adapted_latest.jsonl
[INFO] RPM: total_valid=1155 adapt=578 eval_holdout=577


RPM SA holdout answering (glm4_can_sa_adapted:latest): 100%|██████████| 577/577 [1:34:53<00:00,  9.87s/it]

[INFO] RPM: holdout answers+explanations saved to RPM_sa_qa\llm_answers\rpm_sa_answers_glm4_can_sa_adapted_latest.jsonl


In [ ]:
# Explanations are generated inline with answers in Cell 5.
# This cell is intentionally kept as a no-op for backwards compatibility.
print("[INFO] Inline mode active: llm_answer and llm_explanation are generated together in Cell 5.")



In [ ]:
# Legacy post-processing explanation pass is disabled.
# Explanations are already written in the same JSONL line as llm_answer in Cell 5.
print("[INFO] Skipping legacy explanation pass (already generated inline).")

